# Table C

Table C represents the cycle-level dataset derived from the cleaned ultrasonic sensor readings. Each record corresponds to a single bin filling cycle, defined as the period between two consecutive collection events. This table is used in our final model in predicting the average daily fill growth and supporting collection planning decisions.

## Data Source
The dataset used in this project is the ultrasonic waste bin sensor dataset published on Zenodo:
https://zenodo.org/records/14988663

The dataset contains fill-level measurements captured by ultrasonic sensors installed in waste bins. Sensor timestamps are generated automatically, while collection timestamps are manually entered by service providers through the management system.

For this project, we use the corrected fill files provided by the dataset authors, which include preprocessing steps to address data quality issues present in the raw sensor readings.

## Table C Schema

Table C contains the following columns:

- ContainerID
- Cidx
- start_timestamp
- end_timestamp
- collection_fill_percentage
- cycle_start_month
- cycle_duration_days 
- avg_daily_fill_growth 
- next_cycle_avg_daily_fill_growth

## Field Definitions
| Field | Description |
|------|-------------|
| ContainerID | Unique identifier for each waste bin.|
| CIDX (Cycle Index) | Identifies a single filling cycle between two collection events. All readings with the same CIDX belong to the same continuous filling period. |
| start_timestamp | Timestamp corresponding to the first fill-level reading immediately after a collection event, marking the start of a new cycle. |
| end_timestamp | Timestamp corresponding to the last fill-level reading before the next collection event, marking the end of the cycle.|
| collection_fill_percentage | Fill percentage of the bin at the point of collection, representing how full the bin was when it was emptied. |
| cycle_start_month | Calendar month extracted from the cycle start timestamp, used to capture seasonal or monthly variation in disposal behaviour. |
| cycle_duration_days | Total duration of the fill cycle, measured in days between the start and end timestamps. |
| avg_daily_fill_growth | Average daily increase in fill percentage during the cycle, calculated as collection fill percentage divided by cycle duration. |
|next_cycle_avg_daily_fill_growth | Average daily fill growth observed in the next cycle|


In [1]:
# Imports & Setup
from datasets import load_dataset
from huggingface_hub import list_repo_files
import pandas as pd
import os
import logging
from datasets.utils.logging import disable_progress_bar

# to suppress Hugging Face info messages due to missing yaml metadata
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
disable_progress_bar()

In [ ]:
# Dataset Source
repo_id = "SA61team5/ultrasonic-waste-bin-sensor-raw"

all_files = list_repo_files(repo_id, repo_type="dataset")

# Select corrected fill csv files only
fill_files = sorted(
    f for f in all_files 
    if "_fill_Corrected_with_metrics" in f and f.endswith(".csv")
)

# Limit to 30 bins for ML
fill_files = fill_files[:30]


## Per-bin Data Loading and Cleaning

For each selected bin file, we:
- Load the corrected fill data
- Extract the container identifier from the filename 
- Standardise timestamps
- Rename and retain only relevant columns

In [3]:
all_fill_dfs = []

for file_path in fill_files:
    dataset = load_dataset(repo_id, data_files=file_path)
    df = pd.DataFrame(dataset["train"][:])

    # Extract container ID from filename
    container_id = os.path.basename(file_path).split("_")[1]
    df["ContainerID"] = container_id

    # Parse timestamp and derive month and is_weekend
    df["Timestamp"] = pd.to_datetime(df["Date"])
    df["Month"] = df["Timestamp"].dt.month

    # Rename and clean
    df = df.rename(columns={"Mean": "Fill_percentage"})
    df = df.dropna(subset=["Fill_percentage", "Cidx"])
    
    df = df.dropna(subset=["Fill_percentage"])

    df = df.dropna(subset=["Cidx"])

    df = df.drop(columns=["Max", "Min", "Fill", "Date"])

    cols = ["ContainerID", "Timestamp", "Fill_percentage", "Cidx", "Rec", "Month", ]
    df = df[cols]

    all_fill_dfs.append(df)

final_fill_df = pd.concat(all_fill_dfs, ignore_index=True)

# Sort by bin and time
final_fill_df = final_fill_df.sort_values(by=["ContainerID", "Timestamp"])

# Rename ContainerID to 1 to 30
final_fill_df["ContainerID"] = (
    final_fill_df["ContainerID"]
    .astype("category")
    .cat.codes
    + 1
)

final_fill_df.head()


,ContainerID,Timestamp,Fill_percentage,Cidx,Rec,Month
0,1,2021-01-16 11:30:00,52.5,0.0,1,1
1,1,2021-01-16 12:28:00,52.5,0.0,0,1
2,1,2021-01-16 13:28:00,52.5,0.0,0,1
3,1,2021-01-17 12:34:00,52.5,0.0,0,1
4,1,2021-01-18 12:41:00,52.5,0.0,0,1


# Calculate Cycle Duration Days

In [9]:
# group rows by bin and collection cycle
cycle_df = (
    final_fill_df
    .groupby(["ContainerID", "Cidx"], as_index=False)
    .agg(
        start_timestamp=("Timestamp", "min"),
        end_timestamp=("Timestamp", "max"),
        collection_fill_percentage=("Fill_percentage", "max"),
        cycle_start_month=("Month", "first"),
    )
)

# calculate cycle duration days
cycle_df["cycle_duration_days"] = (
    (cycle_df["end_timestamp"] - cycle_df["start_timestamp"])
    .dt.total_seconds() / (60 * 60 * 24)
).astype(int)

# Remove invalid cycles
cycle_df = cycle_df[cycle_df["cycle_duration_days"] > 0]

cycle_df.head()


,ContainerID,Cidx,start_timestamp,end_timestamp,collection_fill_percentage,cycle_start_month,cycle_duration_days
0,1,0.0,2021-01-16 11:30:00,2021-02-01 17:10:00,100.0,1,16
1,1,2.0,2021-02-01 20:11:00,2021-02-05 14:43:00,44.0,2,3
2,1,4.0,2021-02-06 01:15:00,2021-02-12 22:09:00,72.0,2,6
3,1,5.0,2021-02-12 23:10:00,2021-02-19 16:34:00,100.0,2,6
4,1,6.0,2021-02-20 12:08:00,2021-02-26 19:02:00,100.0,2,6


# Calculate Average Daily Fill Growth per cycle

In [10]:
# Calculate average daily fill growth per cycle
cycle_df["avg_daily_fill_growth"] = (
    cycle_df["collection_fill_percentage"] / cycle_df["cycle_duration_days"]
)

cycle_df.head()

,ContainerID,Cidx,start_timestamp,end_timestamp,collection_fill_percentage,cycle_start_month,cycle_duration_days,avg_daily_fill_growth
0,1,0.0,2021-01-16 11:30:00,2021-02-01 17:10:00,100.0,1,16,6.250000
1,1,2.0,2021-02-01 20:11:00,2021-02-05 14:43:00,44.0,2,3,14.666667
2,1,4.0,2021-02-06 01:15:00,2021-02-12 22:09:00,72.0,2,6,12.000000
3,1,5.0,2021-02-12 23:10:00,2021-02-19 16:34:00,100.0,2,6,16.666667
4,1,6.0,2021-02-20 12:08:00,2021-02-26 19:02:00,100.0,2,6,16.666667


In [6]:
# Forward fill each row with the next cycle average daily fill growth
cycle_df = cycle_df.sort_values(["ContainerID", "Cidx"])

cycle_df["next_cycle_avg_daily_fill_growth"] = (
    cycle_df
    .groupby("ContainerID")["avg_daily_fill_growth"]
    .shift(-1)
)

cycle_df = cycle_df.dropna(subset=["next_cycle_avg_daily_fill_growth"])
cycle_df.head()

,ContainerID,Cidx,start_timestamp,end_timestamp,collection_fill_percentage,cycle_start_month,cycle_duration_days,avg_daily_fill_growth,next_cycle_avg_daily_fill_growth
0,1,0.0,2021-01-16 11:30:00,2021-02-01 17:10:00,100.0,1,16,6.250000,14.666667
1,1,2.0,2021-02-01 20:11:00,2021-02-05 14:43:00,44.0,2,3,14.666667,12.000000
2,1,4.0,2021-02-06 01:15:00,2021-02-12 22:09:00,72.0,2,6,12.000000,16.666667
3,1,5.0,2021-02-12 23:10:00,2021-02-19 16:34:00,100.0,2,6,16.666667,16.666667
4,1,6.0,2021-02-20 12:08:00,2021-02-26 19:02:00,100.0,2,6,16.666667,23.666667


In [7]:
# save as csv file
cycle_df.to_csv("../Cleaned-data/tableC.csv", index=True, index_label="Index")